# 🔫 Phase 1: Weapon & Violence Detection

**Samsung PRISM - Harmful Content Detection Pipeline**

This notebook trains a YOLOv8 model for detecting weapons and dangerous objects in images.

## Detectable Classes
- Guns (pistol, rifle)
- Knives
- Machetes
- Bats
- Alcohol bottles
- Swords

---

**⚠️ IMPORTANT: Enable GPU in Colab**
- Go to `Runtime` → `Change runtime type` → Select `GPU`

## 1️⃣ Setup & Installation

In [ ]:
# Install required packages
!pip install ultralytics>=8.0.200 -q
!pip install roboflow -q
!pip install albumentations -q

# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Import libraries
import os
import yaml
import shutil
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image, display
import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive

# Mount Google Drive for saving models
drive.mount('/content/drive')

## 2️⃣ Dataset Download Instructions

### Option A: Manual Download from Kaggle

Download these datasets and upload to Colab:

| Dataset | Link | Place in |
|---------|------|----------|
| Guns Detection | [kaggle.com/issaisasank/guns-object-detection](https://www.kaggle.com/datasets/issaisasank/guns-object-detection) | `/content/data/guns/` |
| Knife Detection | [kaggle.com/aadityarathod/knife-dataset](https://www.kaggle.com/datasets/aadityarathod/knife-dataset) | `/content/data/knives/` |

### Option B: Use Roboflow (Recommended)

We'll use Roboflow Universe which has pre-annotated weapon datasets in YOLO format.

In [ ]:
# Create data directories
!mkdir -p /content/data/weapons
!mkdir -p /content/runs
!mkdir -p /content/models

In [ ]:
# Option B: Download from Roboflow Universe
# This uses a public weapon detection dataset

from roboflow import Roboflow

# Initialize Roboflow - Get your API key from https://roboflow.com/
# For public datasets, you can use a free account
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # Replace with your key or use public datasets

# Download weapon detection dataset
# Example using a public guns/knives dataset
project = rf.workspace("samsung-prism").project("weapon-detection")
dataset = project.version(1).download("yolov8")

print(f"Dataset downloaded to: {dataset.location}")

In [ ]:
# ALTERNATIVE: If you don't have Roboflow API key,
# upload your Kaggle datasets manually and organize them

# For manual upload, structure should be:
# /content/data/weapons/
#   ├── train/
#   │   ├── images/
#   │   └── labels/
#   ├── valid/
#   │   ├── images/
#   │   └── labels/
#   └── test/
#       ├── images/
#       └── labels/

# Upload your zip files and extract them:
# from google.colab import files
# uploaded = files.upload()  # Upload your dataset zip
# !unzip uploaded_file.zip -d /content/data/weapons/

## 3️⃣ Dataset Configuration

In [ ]:
# Create data.yaml for YOLOv8 training

data_yaml = """
# Weapon Detection Dataset Configuration
path: /content/data/weapons
train: train/images
val: valid/images
test: test/images

# Classes
names:
  0: gun
  1: pistol
  2: rifle
  3: knife
  4: machete
  5: bat
  6: alcohol_bottle
  7: broken_bottle
  8: sword

# Number of classes
nc: 9
"""

# Save configuration
with open('/content/data/weapons/data.yaml', 'w') as f:
    f.write(data_yaml)

print("✓ Created data.yaml configuration")
print(data_yaml)

In [ ]:
# Verify dataset structure
import os

def count_files(directory):
    if os.path.exists(directory):
        return len([f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))])
    return 0

base_path = '/content/data/weapons'
print("Dataset Statistics:")
print("=" * 40)

for split in ['train', 'valid', 'test']:
    img_count = count_files(f'{base_path}/{split}/images')
    lbl_count = count_files(f'{base_path}/{split}/labels')
    print(f"{split.upper():10} | Images: {img_count:5} | Labels: {lbl_count:5}")

## 4️⃣ Data Visualization

In [ ]:
# Visualize sample images with annotations
import cv2
import matplotlib.pyplot as plt
import random

def visualize_sample(image_path, label_path, class_names):
    """Visualize image with bounding box annotations."""
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # Read labels (YOLO format: class x_center y_center width height)
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    x_center, y_center, bw, bh = map(float, parts[1:5])
                    
                    # Convert to pixel coordinates
                    x1 = int((x_center - bw/2) * w)
                    y1 = int((y_center - bh/2) * h)
                    x2 = int((x_center + bw/2) * w)
                    y2 = int((y_center + bh/2) * h)
                    
                    # Draw rectangle
                    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    
                    # Draw label
                    label = class_names.get(cls_id, f"class_{cls_id}")
                    cv2.putText(img, label, (x1, y1-10), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
    
    return img

# Class names mapping
class_names = {
    0: 'gun', 1: 'pistol', 2: 'rifle', 3: 'knife',
    4: 'machete', 5: 'bat', 6: 'alcohol_bottle',
    7: 'broken_bottle', 8: 'sword'
}

# Get sample images
train_images_dir = '/content/data/weapons/train/images'
train_labels_dir = '/content/data/weapons/train/labels'

if os.path.exists(train_images_dir):
    images = os.listdir(train_images_dir)
    samples = random.sample(images, min(4, len(images)))
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    for ax, img_name in zip(axes.flat, samples):
        img_path = os.path.join(train_images_dir, img_name)
        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(train_labels_dir, label_name)
        
        img = visualize_sample(img_path, label_path, class_names)
        ax.imshow(img)
        ax.set_title(img_name)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('/content/sample_annotations.png', dpi=150)
    plt.show()
else:
    print("⚠️ Training data not found. Please upload dataset first.")

## 5️⃣ Model Training

In [ ]:
# Load YOLOv8 pretrained model
# Options: yolov8n (nano), yolov8s (small), yolov8m (medium), yolov8l (large), yolov8x (extra large)

MODEL_SIZE = 'yolov8m'  # Medium - good balance of speed and accuracy

model = YOLO(f'{MODEL_SIZE}.pt')
print(f"✓ Loaded {MODEL_SIZE} pretrained weights")

In [ ]:
# Training configuration
TRAINING_CONFIG = {
    'data': '/content/data/weapons/data.yaml',
    'epochs': 50,                # Number of training epochs (increase for better results)
    'imgsz': 640,                # Image size
    'batch': 16,                 # Batch size (reduce if OOM)
    'device': 0,                 # GPU device
    'workers': 4,                # Data loader workers
    'patience': 10,              # Early stopping patience
    'save': True,                # Save checkpoints
    'save_period': 10,           # Save every N epochs
    'project': '/content/runs',  # Save directory
    'name': 'weapon_detection',  # Run name
    'exist_ok': True,            # Overwrite existing
    'pretrained': True,          # Use pretrained weights
    'optimizer': 'AdamW',        # Optimizer
    'lr0': 0.01,                 # Initial learning rate
    'lrf': 0.01,                 # Final learning rate factor
    'momentum': 0.937,           # SGD momentum
    'weight_decay': 0.0005,      # Weight decay
    'warmup_epochs': 3.0,        # Warmup epochs
    'box': 7.5,                  # Box loss gain
    'cls': 0.5,                  # Class loss gain
    'mosaic': 1.0,               # Mosaic augmentation
    'mixup': 0.15,               # Mixup augmentation
    'hsv_h': 0.015,              # HSV-Hue augmentation
    'hsv_s': 0.7,                # HSV-Saturation augmentation
    'hsv_v': 0.4,                # HSV-Value augmentation
    'flipud': 0.0,               # Flip up-down
    'fliplr': 0.5,               # Flip left-right
}

print("Training Configuration:")
for k, v in TRAINING_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# 🚀 Start Training
print("\n" + "="*50)
print("🚀 STARTING WEAPON DETECTION MODEL TRAINING")
print("="*50 + "\n")

results = model.train(**TRAINING_CONFIG)

print("\n" + "="*50)
print("✓ TRAINING COMPLETED")
print("="*50)

## 6️⃣ Model Evaluation

In [ ]:
# Load best model
best_model_path = '/content/runs/weapon_detection/weights/best.pt'
trained_model = YOLO(best_model_path)

print(f"✓ Loaded best model from: {best_model_path}")

In [ ]:
# Validate on test set
metrics = trained_model.val(
    data='/content/data/weapons/data.yaml',
    split='test',
    batch=16,
    imgsz=640,
    device=0
)

# Display metrics
print("\n" + "="*50)
print("📊 EVALUATION METRICS")
print("="*50)
print(f"mAP@50:      {metrics.box.map50:.4f}")
print(f"mAP@50-95:   {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")
print("="*50)

In [ ]:
# Display training curves
print("\n📈 Training Curves:")

# Results images
results_dir = '/content/runs/weapon_detection'

# Display confusion matrix
if os.path.exists(f'{results_dir}/confusion_matrix.png'):
    display(Image(filename=f'{results_dir}/confusion_matrix.png', width=600))

# Display results summary
if os.path.exists(f'{results_dir}/results.png'):
    display(Image(filename=f'{results_dir}/results.png', width=800))

In [ ]:
# Per-class metrics
print("\n📊 Per-Class Performance:")
print("-" * 50)
print(f"{'Class':<20} {'Precision':<12} {'Recall':<12} {'mAP50':<12}")
print("-" * 50)

class_names_list = ['gun', 'pistol', 'rifle', 'knife', 'machete', 
                    'bat', 'alcohol_bottle', 'broken_bottle', 'sword']

for i, name in enumerate(class_names_list):
    if i < len(metrics.box.p):
        print(f"{name:<20} {metrics.box.p[i]:.4f}       {metrics.box.r[i]:.4f}       {metrics.box.ap50[i]:.4f}")

## 7️⃣ Inference Demo

In [ ]:
# Run inference on test images
def run_inference(model, image_path, conf_threshold=0.5):
    """Run inference on a single image."""
    results = model(image_path, verbose=False)[0]
    
    detections = []
    for box in results.boxes:
        conf = float(box.conf[0])
        if conf >= conf_threshold:
            cls_id = int(box.cls[0])
            cls_name = results.names.get(cls_id, f"class_{cls_id}")
            bbox = box.xyxy[0].cpu().numpy().tolist()
            
            detections.append({
                'class': cls_name,
                'confidence': round(conf, 4),
                'bbox': bbox
            })
    
    return {
        'image': image_path,
        'detected': len(detections) > 0,
        'detections': detections,
        'visualization': results.plot()
    }

print("✓ Inference function defined")

In [ ]:
# Test on sample images
test_images_dir = '/content/data/weapons/test/images'

if os.path.exists(test_images_dir):
    test_images = os.listdir(test_images_dir)
    samples = random.sample(test_images, min(4, len(test_images)))
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    
    for ax, img_name in zip(axes.flat, samples):
        img_path = os.path.join(test_images_dir, img_name)
        result = run_inference(trained_model, img_path)
        
        # Display
        vis_img = cv2.cvtColor(result['visualization'], cv2.COLOR_BGR2RGB)
        ax.imshow(vis_img)
        
        # Title with detections
        if result['detected']:
            detections_str = ', '.join([f"{d['class']} ({d['confidence']:.2f})" 
                                        for d in result['detections']])
            ax.set_title(f"🚨 DETECTED: {detections_str}", fontsize=10, color='red')
        else:
            ax.set_title("✅ No weapons detected", fontsize=10, color='green')
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('/content/inference_results.png', dpi=150)
    plt.show()
else:
    print("⚠️ Test images not found.")

In [ ]:
# Example JSON output format
import json

# Sample output
sample_output = {
    "status": "UNSAFE",
    "detection_type": "weapon",
    "detections": [
        {
            "class": "gun",
            "confidence": 0.92,
            "bbox": {"x1": 100, "y1": 150, "x2": 300, "y2": 400}
        }
    ],
    "explanation": "Weapon detected: gun with 92% confidence"
}

print("\n📋 Sample Output Format:")
print(json.dumps(sample_output, indent=2))

## 8️⃣ Save Model

In [ ]:
# Save best model to Google Drive
drive_path = '/content/drive/MyDrive/samsung_prism/models/weapon_detector'
os.makedirs(drive_path, exist_ok=True)

# Copy best weights
shutil.copy(best_model_path, f'{drive_path}/best.pt')
shutil.copy('/content/runs/weapon_detection/weights/last.pt', f'{drive_path}/last.pt')

# Save training results
shutil.copy('/content/runs/weapon_detection/results.csv', f'{drive_path}/results.csv')

print(f"\n✓ Model saved to: {drive_path}")
print("\nSaved files:")
for f in os.listdir(drive_path):
    size_mb = os.path.getsize(f'{drive_path}/{f}') / (1024*1024)
    print(f"  - {f} ({size_mb:.2f} MB)")

In [ ]:
# Export to other formats (optional)
# Export to ONNX for deployment
# trained_model.export(format='onnx')

# Export to TensorRT for faster inference
# trained_model.export(format='engine')

print("\n✓ Weapon Detection Model Training Complete!")
print("\nNext Steps:")
print("  1. Run 02_text_classification.ipynb")
print("  2. Run 03_logo_detection.ipynb")
print("  3. Run 04_unified_pipeline.ipynb for end-to-end inference")

---

## Summary

| Metric | Value |
|--------|-------|
| Model | YOLOv8m |
| Classes | 9 (weapons) |
| mAP@50 | [See above] |
| Precision | [See above] |
| Recall | [See above] |

**Model saved to:** `/content/drive/MyDrive/samsung_prism/models/weapon_detector/best.pt`